<a href="https://colab.research.google.com/github/Lorenzito/ProgramacionParaAnaliticaDescriptivayPredicitva/blob/main/S06_Actividad_01_NumPy_273524.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 06 — Actividad calificable: NumPy y preparación de datos

# Actividad calificada — NumPy y preparación de datos

**Programación para Analítica Descriptiva y Predictiva**

Maestría en Inteligencia Artificial y Analítica de Datos — UACJ

---


**Nombre completo:** Lorenzo Varela Ollervides
**Matrícula:** 273524

---

**Entrega:** individual, en este notebook de Google Colab (comparte el enlace con permiso de edición para el docente).
**Plazo:** 7 días naturales a partir de la publicación de esta actividad.
**Penalización por entrega tardía:** 5% por día, hasta 7 días; después de eso, la actividad no se recibe.
**Valor total:** 100 puntos.

### Herramientas permitidas

Esta actividad evalúa lo visto hasta la Sesión 6: expresiones regulares (`re`), comprensión de listas, `map()`, `lambda`, y NumPy. **No uses pandas ni scikit-learn** — todavía no se han visto en el curso, y su uso no sumará puntos aunque el resultado sea correcto.

### Cómo se evalúa

Cada ejercicio indica su valor en puntos. Se evalúa el resultado correcto **y** el uso apropiado de las herramientas indicadas en cada enunciado — una solución correcta que ignore la herramienta pedida (por ejemplo, usar un bucle `for` donde se pide `map()` o una operación vectorizada) no obtiene el puntaje completo. Comenta tu código donde el paso no sea evidente.

## Librerías

Ejecuta esta celda antes de empezar.

In [92]:
import re
import numpy as np

## Datos de partida

Una red de 4 estaciones meteorológicas registró temperatura y humedad durante 5 días. Los datos llegaron en un formato de texto crudo, como ocurre frecuentemente al recibir información de sensores o sistemas externos. Ejecuta la siguiente celda para cargar las lecturas — **no la modifiques**.

In [93]:
lecturas_crudas = [
    "EST-01 | Dia1 | Temp: 22.3C | Hum: 60%",
    "EST-01 | Dia2 | Temp: 19.8C | Hum: 64%",
    "EST-01 | Dia3 | Temp: 24.1C | Hum: 58%",
    "EST-01 | Dia4 | Temp: 26.5C | Hum: 55%",
    "EST-01 | Dia5 | Temp: 21.0C | Hum: 63%",
    "EST-02 | Dia1 | Temp: 24.0C | Hum: 70%",
    "EST-02 | Dia2 | Temp: 23.5C | Hum: 72%",
    "EST-02 | Dia3 | Temp: 27.2C | Hum: 65%",
    "EST-02 | Dia4 | Temp: 20.1C | Hum: 74%",
    "EST-02 | Dia5 | Temp: 22.8C | Hum: 69%",
    "EST-03 | Dia1 | Temp: 18.5C | Hum: 80%",
    "EST-03 | Dia2 | Temp: 19.2C | Hum: 78%",
    "EST-03 | Dia3 | Temp: 20.0C | Hum: 77%",
    "EST-03 | Dia4 | Temp: 21.5C | Hum: 75%",
    "EST-03 | Dia5 | Temp: 19.9C | Hum: 79%",
    "EST-04 | Dia1 | Temp: 27.8C | Hum: 50%",
    "EST-04 | Dia2 | Temp: 26.3C | Hum: 52%",
    "EST-04 | Dia3 | Temp: 25.9C | Hum: 53%",
    "EST-04 | Dia4 | Temp: 24.0C | Hum: 56%",
    "EST-04 | Dia5 | Temp: 28.1C | Hum: 49%",
]

print(len(lecturas_crudas), "lecturas cargadas")
print(lecturas_crudas[0])

20 lecturas cargadas
EST-01 | Dia1 | Temp: 22.3C | Hum: 60%


---

## Ejercicio 1 — Extracción con expresiones regulares (10 pts)

Usando `re` y comprensión de listas, extrae de cada cadena en `lecturas_crudas` el identificador de estación (por ejemplo `'EST-01'`) y el valor de temperatura como texto (por ejemplo `'22.3'`, sin la `C`). El resultado debe ser una lista de tuplas `(estacion, temperatura_texto)`, en el mismo orden que `lecturas_crudas`.

*Por qué importa: en la práctica, los datos casi nunca llegan ya estructurados — extraerlos de texto libre es el primer paso de cualquier pipeline de análisis.*

In [94]:
# Tu código aquí
expresion = r'(EST-\d+) \| Dia\d+ \| Temp: (\d+\.\d+)C'
extraidos = [ re.findall(expresion, cadena)[0] for cadena in lecturas_crudas ]

print(extraidos)

[('EST-01', '22.3'), ('EST-01', '19.8'), ('EST-01', '24.1'), ('EST-01', '26.5'), ('EST-01', '21.0'), ('EST-02', '24.0'), ('EST-02', '23.5'), ('EST-02', '27.2'), ('EST-02', '20.1'), ('EST-02', '22.8'), ('EST-03', '18.5'), ('EST-03', '19.2'), ('EST-03', '20.0'), ('EST-03', '21.5'), ('EST-03', '19.9'), ('EST-04', '27.8'), ('EST-04', '26.3'), ('EST-04', '25.9'), ('EST-04', '24.0'), ('EST-04', '28.1')]


## Explicación del código

El patrón `expresion` tiene dos grupos de captura entre paréntesis:

- `(EST-\d+)` → captura el identificador de estación, por ejemplo `'EST-01'`
- `(\d+\.\d+)` → captura el valor de temperatura, por ejemplo `'22.3'`

Para cada `cadena`, `re.findall(expresion, cadena)` devuelve una **lista de tuplas** con lo capturado por esos grupos. Como cada cadena solo tiene una coincidencia, el resultado es una lista con una sola tupla, por ejemplo `[('EST-01', '22.3')]`. El `[0]` extrae esa tupla de la lista.

La comprensión de listas repite este proceso para cada elemento de `lecturas_crudas`, generando la lista final de tuplas `(estacion, temperatura_texto)` en el mismo orden original.

---

## Ejercicio 2 — Construcción del arreglo numérico (10 pts)

A partir de las temperaturas extraídas en el Ejercicio 1 (que están en texto), usa `map()` junto con una conversión a `float` para construir un `ndarray` de tipo numérico. Verifica e imprime su `dtype`.

*Por qué importa: conectar la extracción de texto con un tipo de dato numérico utilizable es un paso que se repite constantemente al preparar datos.*

In [95]:
# Tu código aquí
temperaturas = np.array(list(map(lambda temp: float(temp[1]), extraidos)))
print(temperaturas)
print(temperaturas.dtype)

[22.3 19.8 24.1 26.5 21.  24.  23.5 27.2 20.1 22.8 18.5 19.2 20.  21.5
 19.9 27.8 26.3 25.9 24.  28.1]
float64


---

## Ejercicio 3 — Organización en una matriz 2D (10 pts)

Reorganiza el arreglo de temperaturas del Ejercicio 2 en una matriz de 4 filas (una por estación) × 5 columnas (una por día) usando `reshape()`. Los datos ya están ordenados por estación en `lecturas_crudas`, así que no necesitas reordenarlos manualmente. Reporta `shape`, `ndim`, `size` y `dtype` de la matriz resultante.

In [96]:
# Tu código aquí
matriz_temp = temperaturas.reshape(4,5)
print(matriz_temp)
print("-" * 30)
print("shape", matriz_temp.shape)
print("ndim", matriz_temp.ndim)
print("size", matriz_temp.size)
print("dtype",matriz_temp.dtype)


[[22.3 19.8 24.1 26.5 21. ]
 [24.  23.5 27.2 20.1 22.8]
 [18.5 19.2 20.  21.5 19.9]
 [27.8 26.3 25.9 24.  28.1]]
------------------------------
shape (4, 5)
ndim 2
size 20
dtype float64


---

## Ejercicio 4 — Indexado, slicing y transposición (10 pts)

A partir de `matriz_temp`:

1. Extrae la fila completa correspondiente a `EST-03`.
2. Extrae la columna completa correspondiente al Día 4.
3. Extrae la submatriz de las primeras 2 estaciones durante los primeros 3 días.
4. Transpón la matriz con `.T` e imprime su nueva forma (`shape`). En una celda de texto (Markdown) o un comentario, explica en qué situación tendría sentido trabajar con los días como filas y las estaciones como columnas.

In [97]:
# Tu código aquí
estacion3 = matriz_temp[2, :]
print("EST-03\t", estacion3)

dia4 = matriz_temp[:, 3]
print("Día 4\t", dia4)

submatriz = matriz_temp[:2, :3]
print("Submatriz \n", submatriz)

matriz_transpuesta = matriz_temp.T
print("Matriz transpuesta \n", matriz_transpuesta)


EST-03	 [18.5 19.2 20.  21.5 19.9]
Día 4	 [26.5 20.1 21.5 24. ]
Submatriz 
 [[22.3 19.8 24.1]
 [24.  23.5 27.2]]
Matriz transpuesta 
 [[22.3 24.  18.5 27.8]
 [19.8 23.5 19.2 26.3]
 [24.1 27.2 20.  25.9]
 [26.5 20.1 21.5 24. ]
 [21.  22.8 19.9 28.1]]


Trabajar con los dias como fila y las estaciones como columnas tiene sentido cuando lo que nos interesa hacer un analisis de las diferentes temperaturas

---

## Ejercicio 5 — Máscaras booleanas, verificación y fancy indexing (15 pts)

1. Usa una máscara booleana sobre `matriz_temp` para identificar qué lecturas superan los 25°C.
2. Usando `any()`, verifica si **alguna** estación tuvo, en algún día, una lectura mayor a 25°C.
3. Usando `all()`, verifica si la estación `EST-04` (fila correspondiente) tuvo **todos** sus días por encima de los 24°C.
4. Usando fancy indexing, reordena las filas de `matriz_temp` según la lista de prioridad `[2, 0, 3, 1]` (es decir, primero EST-03, luego EST-01, EST-04 y EST-02).

*Por qué importa: filtrar por condición, verificar rápidamente supuestos sobre los datos, y reordenar según un criterio externo son operaciones cotidianas al priorizar qué datos revisar primero.*

In [98]:
# Tu código aquí
print("Temperaturas superiores a 25°C")
print(matriz_temp[matriz_temp > 25])

print("-" * 30)
print("Al menos una estación tuvo una lectura mayor a 25°C")
print((matriz_temp > 25).any())
print("-" * 30)
print("La estación EST-04 tuvo todos sus días por encima de los 24°C")
print((matriz_temp[3,:] > 15).all())
print("-" * 30)
print("Reordenado según la lista de prioridad")
print(matriz_temp[[2, 0, 3, 1], :])


Temperaturas superiores a 25°C
[26.5 27.2 27.8 26.3 25.9 28.1]
------------------------------
Al menos una estación tuvo una lectura mayor a 25°C
True
------------------------------
La estación EST-04 tuvo todos sus días por encima de los 24°C
True
------------------------------
Reordenado según la lista de prioridad
[[18.5 19.2 20.  21.5 19.9]
 [22.3 19.8 24.1 26.5 21. ]
 [27.8 26.3 25.9 24.  28.1]
 [24.  23.5 27.2 20.1 22.8]]


---

## Ejercicio 6 — Broadcasting y `np.where()` (15 pts)

Cada estación tiene un descalibre de sensor conocido, dado por este arreglo (en el mismo orden que las filas de `matriz_temp`: EST-01, EST-02, EST-03, EST-04):

```python
descalibre = np.array([-0.5, 1.2, 0.0, -1.0])
```

1. Usa broadcasting para aplicar la corrección correspondiente a cada fila de `matriz_temp` (cada estación se ajusta con su propio valor de `descalibre`). Guarda el resultado en una variable nueva, por ejemplo `matriz_corregida`. **Pista:** si intentas sumar `matriz_temp + descalibre` directamente obtendrás un error de formas incompatibles — el broadcasting solo funciona si `descalibre` tiene la forma `(4, 1)` en lugar de `(4,)`, para que cada valor se alinee con una fila completa. Usa `reshape()` (ya lo conoces del Bloque 2) para darle esa forma antes de sumar.
2. Sobre `matriz_corregida`, usa `np.where()` para generar una matriz paralela de etiquetas de texto: `'alta'` si la lectura corregida supera los 25°C, `'normal'` en caso contrario.

In [99]:
descalibre = np.array([-0.5, 1.2, 0.0, -1.0])

# Tu código aquí

matriz_corregida = matriz_temp + descalibre.reshape(4,1)
print(matriz_corregida)
print("-" * 30)
etiquetas = np.where(matriz_corregida > 25, 'alta', 'normal')
print(etiquetas)

[[21.8 19.3 23.6 26.  20.5]
 [25.2 24.7 28.4 21.3 24. ]
 [18.5 19.2 20.  21.5 19.9]
 [26.8 25.3 24.9 23.  27.1]]
------------------------------
[['normal' 'normal' 'normal' 'alta' 'normal']
 ['alta' 'normal' 'alta' 'normal' 'normal']
 ['normal' 'normal' 'normal' 'normal' 'normal']
 ['alta' 'alta' 'normal' 'normal' 'alta']]


---

## Ejercicio 7 — Agregaciones por eje (15 pts)

Usando `matriz_corregida` del Ejercicio 6:

1. Calcula el promedio y la desviación estándar de temperatura **por estación** (usa el `axis` correspondiente).
2. Calcula el promedio de temperatura **por día** (usa el `axis` correspondiente).
3. Sin usar funciones no vistas en el curso (por ejemplo, sin `argmax`), determina **qué día** tuvo el promedio más alto entre estaciones. Sugerencia: obtén el valor máximo de los promedios por día con `.max()`, y compáralo con el arreglo de promedios por día usando `==` para generar una máscara booleana — la posición con `True` te dice qué día fue.

In [100]:
# Tu código aquí
# Axis 0 toma columnas, Axis 1 toma filas
print("Promedio por estación")
print(matriz_corregida.mean(axis=1))
print("-" * 30)
print("Desviación estándar por estación")
print(matriz_corregida.std(axis=1))
print("-" * 30)
print("Promedio por día")
print(matriz_corregida.mean(axis=0))
print("-" * 30)
print("Día con promedio más alto")
# Utilizamos una comprensión con enumerate para obtener el indice y una condicional para ver el valor true sobre la máscara booleana
# Le sumamos 1 ya que los indices inician desde 0 para obtener el dia real
dia = [indice + 1 for indice, val in enumerate((matriz_corregida.mean(axis=0) == matriz_corregida.mean(axis=0).max())) if val == True ][0]
print(dia)

Promedio por estación
[22.24 24.72 19.82 25.42]
------------------------------
Desviación estándar por estación
[2.36016949 2.27982455 0.99879928 1.4743134 ]
------------------------------
Promedio por día
[23.075 22.125 24.225 22.95  22.875]
------------------------------
Día con promedio más alto
3


---

## Ejercicio 8 — Síntesis: resumen por estación (15 pts)

Usando `map()` y/o comprensión de listas, genera una lista de cadenas de resumen, una por estación, que combine:

- El identificador de estación extraído en el Ejercicio 1.
- El promedio de temperatura corregida de esa estación (Ejercicio 7), formateado a un decimal.
- Su clasificación predominante (`'alta'` o `'normal'`) — usa `np.unique()` sobre las etiquetas de esa estación (del Ejercicio 6) para encontrar cuál aparece con mayor frecuencia entre las dos.

Formato esperado por cadena: `"EST-01: 23.4°C promedio — clasificación predominante: normal"`

Finalmente, ordena la lista de resúmenes de mayor a menor promedio de temperatura usando `sorted()` con `key` y `lambda` (como en la Sesión 4/5), e imprime el resultado ordenado.

In [101]:
# Tu código aquí

# Extraemos los identificadores únicos de estación desde 'extraidos' (Ejercicio 1).
# np.unique() elimina duplicados y los devuelve ordenados. Aquí NO agregamos ':',
# se deja el identificador "limpio" (ej. "EST-01") y el ':' se agrega directamente
# al construir el resumen más adelante.
identificadores = np.unique(np.array([f'{identificador}' for identificador, temp in extraidos]))

In [102]:
# Calculamos el promedio de temperatura corregida por estación (axis=1 = promedio por fila)
# y lo redondeamos a 1 decimal con np.round(), dejándolo como float (no como cadena de texto),
# para aplicar el formato de presentación más adelante, al construir cada resumen.
promedios = np.round(matriz_corregida.mean(axis=1), 1)

In [103]:
# Para cada estación (cada 'fila' de etiquetas del Ejercicio 6), determinamos su clasificación
# predominante ('alta' o 'normal') sin usar argmax:
#   1. np.unique(fila, return_counts=True) devuelve los valores únicos y cuántas veces aparece cada uno.
#   2. Con una máscara booleana (conteos == conteos.max()) nos quedamos con el/los valor(es) de mayor frecuencia.
#   3. [0] al final selecciona el primero, por si hubiera empate entre 'alta' y 'normal'.
clasificaciones = np.array([
    np.unique(fila, return_counts=True)[0][
        np.unique(fila, return_counts=True)[1] == np.unique(fila, return_counts=True)[1].max()
    ][0]
    for fila in etiquetas
])

In [104]:
# Combinamos identificador, promedio y clasificación en una sola cadena de resumen por estación,
# usando map() + lambda sobre zip() para emparejar los tres arrays elemento a elemento.
# El ':' después del identificador, el formato ".1f" y el símbolo "°C" se aplican todos aquí,
# al momento de construir cada cadena.
resumenes = list(map(
    lambda datos: f"{datos[0]}: {datos[1]:.1f}°C promedio — clasificación predominante: {datos[2]}",
    zip(identificadores, promedios, clasificaciones)
))

print(resumenes)

['EST-01: 22.2°C promedio — clasificación predominante: normal', 'EST-02: 24.7°C promedio — clasificación predominante: normal', 'EST-03: 19.8°C promedio — clasificación predominante: normal', 'EST-04: 25.4°C promedio — clasificación predominante: alta']


In [105]:
# Ordenamos los resúmenes de mayor a menor promedio de temperatura.
# Como el promedio quedó embebido en el texto, lo extraemos con split():
#   - resumen.split(":")[1]  -> " 23.4°C promedio — ..."
#   - .split("°C")[0]        -> " 23.4"
#   - float(...)             -> 23.4 (convertido a número para poder comparar/ordenar)
resumenes_ordenados = sorted(
    resumenes,
    key=lambda resumen: float(resumen.split(":")[1].split("°C")[0]),
    reverse=True
)

print("Resúmenes ordenados de mayor a menor promedio:")
for r in resumenes_ordenados:
    print(r)

Resúmenes ordenados de mayor a menor promedio:
EST-04: 25.4°C promedio — clasificación predominante: alta
EST-02: 24.7°C promedio — clasificación predominante: normal
EST-01: 22.2°C promedio — clasificación predominante: normal
EST-03: 19.8°C promedio — clasificación predominante: normal


---

## Antes de entregar

- Verifica que el notebook completo corra de principio a fin sin errores (**Entorno de ejecución → Ejecutar todas**).
- Confirma que no usaste pandas ni scikit-learn.
- Comparte el notebook con permiso de **edición** para el docente.
- Revisa que tu nombre y número de control estén en la primera celda.